In [ ]:
import subprocess, os, shutil

REPO_URL = "https://github.com/safety-research/legibility.git"
WORK_DIR = "/workspace/18-4-2026"

# Clone or pull latest (fetch + reset to ensure we have the newest commit)
if not os.path.exists(os.path.join(WORK_DIR, ".git")):
    subprocess.run(["git", "clone", REPO_URL, WORK_DIR], check=True)
else:
    subprocess.run(["git", "-C", WORK_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", WORK_DIR, "reset", "--hard", "origin/main"], check=True)

# Install git-lfs if not available, then pull LFS files
if shutil.which("git-lfs") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=False)
    subprocess.run(["apt-get", "install", "-y", "-qq", "git-lfs"], check=False)
    subprocess.run(["git", "lfs", "install"], check=False)
subprocess.run(["git", "-C", WORK_DIR, "lfs", "pull"], check=False)

# Install dependencies
req_path = os.path.join(WORK_DIR, "requirements.txt")
if os.path.exists(req_path):
    subprocess.run(["pip", "install", "-q", "-r", req_path], check=True)
else:
    print(f"WARNING: {req_path} not found, skipping pip install")

# Set working directory to notebooks/ so Path.cwd().parent resolves to repo root
os.chdir(os.path.join(WORK_DIR, "notebooks"))

# NB1: Generator Activation Extraction (H200 GPU)

Load G3 (QwQ-32B) and G1 (DeepSeek-R1-Distill-Qwen-32B) sequentially on H200,
extract residual-stream activations for Phase 2 analysis.

**Outputs:**
- `activations/{G1,G3}_last_token/` -- last-token pooled activations at every 4th layer
- `activations/{G1,G3}_question_token/` -- pre-`<think>` position activations (for Exp B)
- `activations/{G1,G3}_full_seq/` -- full-sequence activations at 8 selected layers (for CKA)

**GPU time:** ~3 hours (1.5h per model, sequential)

In [ ]:
import sys
import json
import gc
import torch
import numpy as np
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from phase2_utils import (
    load_phase1_results, get_labeled_cots, join_cots_with_labels,
    get_within_question_pairs, load_model, extract_activations,
    extract_activations_at_position, save_activations, load_activations,
    get_default_layer_indices, print_phase1_summary,
    LOCAL_MODELS, ACTIVATIONS_DIR,
)

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Load Phase 1 results and CoT texts
print_phase1_summary()

# Get all classified (non-filtered) CoTs with text for G1 and G3
# We extract activations for ALL classified samples (leaked + legible + illegible)
# so probes can be trained on any split
all_cots_g1 = join_cots_with_labels(generator_ids=["G1"])
all_cots_g3 = join_cots_with_labels(generator_ids=["G3"])

print(f"G1 CoTs: {len(all_cots_g1)}")
print(f"G3 CoTs: {len(all_cots_g3)}")

# Sort by sample_id for reproducible ordering
all_cots_g1 = sorted(all_cots_g1, key=lambda x: (x['sample_id'], x['epoch']))
all_cots_g3 = sorted(all_cots_g3, key=lambda x: (x['sample_id'], x['epoch']))

In [ ]:
# Verify CoT lengths and spot-check a few samples
for name, cots in [("G1", all_cots_g1), ("G3", all_cots_g3)]:
    lengths = [len(c['cot_text']) for c in cots]
    labels = [c['label'] for c in cots]
    from collections import Counter
    print(f"\n{name}:")
    print(f"  Label distribution: {dict(Counter(labels))}")
    print(f"  CoT length: min={min(lengths)}, median={np.median(lengths):.0f}, max={max(lengths)}")
    # Spot check
    sample = cots[0]
    print(f"  Sample: {sample['sample_id']} label={sample['label']}")
    print(f"  CoT preview: {sample['cot_text'][:200]}...")

## G3: QwQ-32B Activation Extraction

In [ ]:
# Check if all G3 outputs already exist (skip model load if so)
g3_outputs = ["G3_last_token", "G3_question_token", "G3_full_seq"]
g3_all_cached = all((ACTIVATIONS_DIR / name / "metadata.json").exists() for name in g3_outputs)

if g3_all_cached:
    print("CACHED: All G3 outputs exist, skipping G3 model load")
    model_g3 = tok_g3 = None
    n_layers_g3 = None
else:
    # Load G3 (QwQ-32B) -- ~64GB VRAM at bf16
    model_g3, tok_g3 = load_model("G3")
    n_layers_g3 = model_g3.config.num_hidden_layers
    print(f"G3 layers: {n_layers_g3}, hidden_dim: {model_g3.config.hidden_size}")

In [ ]:
output_dir = ACTIVATIONS_DIR / "G3_last_token"
if (output_dir / "metadata.json").exists():
    print(f"CACHED: {output_dir} already exists, skipping extraction")
    g3_last_token = load_activations(output_dir)
else:
    # Extract last-token activations at every 4th layer + last
    layer_indices = get_default_layer_indices(n_layers_g3, stride=4)
    print(f"Extracting at layers: {layer_indices}")

    g3_texts = [c['cot_text'] for c in all_cots_g3]
    g3_last_token = extract_activations(
        model_g3, tok_g3, g3_texts,
        layer_indices=layer_indices,
        pooling="last_token",
        batch_size=1,
        max_length=16384,
    )

    # Verify shapes
    for layer_idx, acts in g3_last_token.items():
        assert acts.shape[0] == len(g3_texts), f"Layer {layer_idx}: expected {len(g3_texts)}, got {acts.shape[0]}"
        assert not np.any(np.isnan(acts)), f"Layer {layer_idx}: NaN detected"
        assert not np.any(np.isinf(acts)), f"Layer {layer_idx}: Inf detected"
        print(f"  Layer {layer_idx}: shape={acts.shape}, norm={np.linalg.norm(acts, axis=1).mean():.2f}")

    # Save with metadata
    metadata_g3 = {
        "model": "G3",
        "hf_id": LOCAL_MODELS["G3"]["hf_id"],
        "pooling": "last_token",
        "n_samples": len(g3_texts),
        "layer_indices": layer_indices,
        "sample_ids": [(c['sample_id'], c['generator_id'], c['epoch'], c['label']) for c in all_cots_g3],
    }
    save_activations(g3_last_token, output_dir, metadata=metadata_g3)

In [ ]:
output_dir = ACTIVATIONS_DIR / "G3_question_token"
if (output_dir / "metadata.json").exists():
    print(f"CACHED: {output_dir} already exists, skipping extraction")
    g3_question_token = load_activations(output_dir)
else:
    # Extract pre-<think> activations for Experiment B
    # Build input texts: question + <think> tag (the model sees this before generating CoT)
    layer_indices = get_default_layer_indices(n_layers_g3, stride=4)
    g3_question_texts = []
    for c in all_cots_g3:
        # Reconstruct the input as the model would have seen it
        g3_question_texts.append(c['input'] + "\n<think>")

    g3_question_token = extract_activations_at_position(
        model_g3, tok_g3, g3_question_texts,
        position="pre_think",
        layer_indices=layer_indices,
        max_length=4096,  # Questions are short
    )

    # Verify
    for layer_idx, acts in g3_question_token.items():
        assert acts.shape[0] == len(g3_question_texts)
        assert not np.any(np.isnan(acts))
        print(f"  Layer {layer_idx}: shape={acts.shape}")

    metadata_g3 = {
        "model": "G3",
        "hf_id": LOCAL_MODELS["G3"]["hf_id"],
        "pooling": "pre_think_position",
        "n_samples": len(g3_question_texts),
        "layer_indices": layer_indices,
        "sample_ids": [(c['sample_id'], c['generator_id'], c['epoch'], c['label']) for c in all_cots_g3],
    }
    save_activations(g3_question_token, output_dir, metadata=metadata_g3)

In [ ]:
output_dir = ACTIVATIONS_DIR / "G3_full_seq"
if (output_dir / "metadata.json").exists():
    print(f"CACHED: {output_dir} already exists, skipping extraction")
    g3_full_seq = load_activations(output_dir)
else:
    # Extract full-sequence activations at 8 selected layers for CKA analysis
    # Only for legible + illegible samples (not answer-leaked) to save space
    cka_layer_indices = [0, 8, 16, 24, 32, 40, 48, 63]

    g3_non_leaked = [c for c in all_cots_g3 if c['label'] != 'ANSWER_LEAKED']
    g3_non_leaked_texts = [c['cot_text'] for c in g3_non_leaked]
    print(f"Extracting full-seq for {len(g3_non_leaked_texts)} non-leaked G3 CoTs")
    print(f"Layers: {cka_layer_indices}")

    g3_full_seq = extract_activations(
        model_g3, tok_g3, g3_non_leaked_texts,
        layer_indices=cka_layer_indices,
        pooling="full",
        batch_size=1,
        max_length=16384,
    )

    # Verify
    for layer_idx, acts_list in g3_full_seq.items():
        print(f"  Layer {layer_idx}: {len(acts_list)} sequences, "
              f"seq_lens=[{acts_list[0].shape[0]}..{acts_list[-1].shape[0]}]")

    metadata_g3_full = {
        "model": "G3",
        "pooling": "full_sequence",
        "n_samples": len(g3_non_leaked_texts),
        "layer_indices": cka_layer_indices,
        "sample_ids": [(c['sample_id'], c['generator_id'], c['epoch'], c['label']) for c in g3_non_leaked],
    }
    save_activations(g3_full_seq, output_dir, metadata=metadata_g3_full)

In [ ]:
# Unload G3 before loading G1
if model_g3 is not None:
    del model_g3, tok_g3
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory after unload: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## G1: DeepSeek-R1-Distill-Qwen-32B Activation Extraction

In [ ]:
# Check if all G1 outputs already exist (skip model load if so)
g1_outputs = ["G1_last_token", "G1_question_token", "G1_full_seq"]
g1_all_cached = all((ACTIVATIONS_DIR / name / "metadata.json").exists() for name in g1_outputs)

if g1_all_cached:
    print("CACHED: All G1 outputs exist, skipping G1 model load")
    model_g1 = tok_g1 = None
    n_layers_g1 = None
else:
    # Load G1 (DeepSeek-R1-Distill-Qwen-32B) -- ~64GB VRAM at bf16
    model_g1, tok_g1 = load_model("G1")
    n_layers_g1 = model_g1.config.num_hidden_layers
    print(f"G1 layers: {n_layers_g1}, hidden_dim: {model_g1.config.hidden_size}")

In [ ]:
output_dir = ACTIVATIONS_DIR / "G1_last_token"
if (output_dir / "metadata.json").exists():
    print(f"CACHED: {output_dir} already exists, skipping extraction")
    g1_last_token = load_activations(output_dir)
else:
    # Extract last-token activations
    layer_indices_g1 = get_default_layer_indices(n_layers_g1, stride=4)
    print(f"Extracting at layers: {layer_indices_g1}")

    g1_texts = [c['cot_text'] for c in all_cots_g1]
    g1_last_token = extract_activations(
        model_g1, tok_g1, g1_texts,
        layer_indices=layer_indices_g1,
        pooling="last_token",
        batch_size=1,
        max_length=16384,
    )

    # Verify shapes
    for layer_idx, acts in g1_last_token.items():
        assert acts.shape[0] == len(g1_texts)
        assert not np.any(np.isnan(acts))
        assert not np.any(np.isinf(acts))
        print(f"  Layer {layer_idx}: shape={acts.shape}, norm={np.linalg.norm(acts, axis=1).mean():.2f}")

    metadata_g1 = {
        "model": "G1",
        "hf_id": LOCAL_MODELS["G1"]["hf_id"],
        "pooling": "last_token",
        "n_samples": len(g1_texts),
        "layer_indices": list(layer_indices_g1),
        "sample_ids": [(c['sample_id'], c['generator_id'], c['epoch'], c['label']) for c in all_cots_g1],
    }
    save_activations(g1_last_token, output_dir, metadata=metadata_g1)

In [ ]:
output_dir = ACTIVATIONS_DIR / "G1_question_token"
if (output_dir / "metadata.json").exists():
    print(f"CACHED: {output_dir} already exists, skipping extraction")
    g1_question_token = load_activations(output_dir)
else:
    # Extract pre-<think> activations for Experiment B
    layer_indices_g1 = get_default_layer_indices(n_layers_g1, stride=4)
    g1_question_texts = [c['input'] + "\n<think>" for c in all_cots_g1]

    g1_question_token = extract_activations_at_position(
        model_g1, tok_g1, g1_question_texts,
        position="pre_think",
        layer_indices=layer_indices_g1,
        max_length=4096,
    )

    for layer_idx, acts in g1_question_token.items():
        assert acts.shape[0] == len(g1_question_texts)
        assert not np.any(np.isnan(acts))
        print(f"  Layer {layer_idx}: shape={acts.shape}")

    metadata_g1_q = {
        "model": "G1",
        "hf_id": LOCAL_MODELS["G1"]["hf_id"],
        "pooling": "pre_think_position",
        "n_samples": len(g1_question_texts),
        "layer_indices": list(layer_indices_g1),
        "sample_ids": [(c['sample_id'], c['generator_id'], c['epoch'], c['label']) for c in all_cots_g1],
    }
    save_activations(g1_question_token, output_dir, metadata=metadata_g1_q)

In [ ]:
output_dir = ACTIVATIONS_DIR / "G1_full_seq"
if (output_dir / "metadata.json").exists():
    print(f"CACHED: {output_dir} already exists, skipping extraction")
    g1_full_seq = load_activations(output_dir)
else:
    # Full-sequence activations for CKA (non-leaked only)
    cka_layer_indices = [0, 8, 16, 24, 32, 40, 48, 63]

    g1_non_leaked = [c for c in all_cots_g1 if c['label'] != 'ANSWER_LEAKED']
    g1_non_leaked_texts = [c['cot_text'] for c in g1_non_leaked]
    print(f"Extracting full-seq for {len(g1_non_leaked_texts)} non-leaked G1 CoTs")

    g1_full_seq = extract_activations(
        model_g1, tok_g1, g1_non_leaked_texts,
        layer_indices=cka_layer_indices,
        pooling="full",
        batch_size=1,
        max_length=16384,
    )

    for layer_idx, acts_list in g1_full_seq.items():
        print(f"  Layer {layer_idx}: {len(acts_list)} sequences")

    metadata_g1_full = {
        "model": "G1",
        "pooling": "full_sequence",
        "n_samples": len(g1_non_leaked_texts),
        "layer_indices": cka_layer_indices,
        "sample_ids": [(c['sample_id'], c['generator_id'], c['epoch'], c['label']) for c in g1_non_leaked],
    }
    save_activations(g1_full_seq, output_dir, metadata=metadata_g1_full)

In [ ]:
# Final cleanup
if model_g1 is not None:
    del model_g1, tok_g1
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory after cleanup: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
print("NB1 complete.")

In [ ]:
# Summary of saved files
import os
for subdir in ['G1_last_token', 'G1_question_token', 'G1_full_seq',
               'G3_last_token', 'G3_question_token', 'G3_full_seq']:
    path = ACTIVATIONS_DIR / subdir
    if path.exists():
        total_size = sum(f.stat().st_size for f in path.rglob('*') if f.is_file())
        print(f"  {subdir}: {total_size / 1e9:.2f} GB")
    else:
        print(f"  {subdir}: NOT FOUND")